# PD03 · Integrar EVA con DIVIPOLA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es comprobar una unión territorial y calcular un rendimiento con el denominador adecuado.

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD03_integracion_territorial.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD03_integracion_territorial.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 7, 9, 10 y la receta de razón agregada de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD03"
archivos = ['eva.csv', 'divipola.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)
divipola = pd.read_csv(RAW / "divipola.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03


## 1. Dimensión territorial · 10 minutos

Crear `codigo_municipio` desde `Código Municipio` de DIVIPOLA. Comprobar cinco cifras y unicidad. Contar los valores de `Tipo: Municipio / Isla / Área no municipalizada`. Construir `dimension` con código, nombre departamental, nombre municipal y clasificación territorial.

📘 **Apoyo:** `.str.strip`, `.str.fullmatch(r"[0-9]{5}")`, `is_unique` y `value_counts`.

✅ **Comprobación:** La dimensión conserva las 1.122 entidades y sus tres tipos. La columna de código es única y no contiene vacíos.

In [2]:
# Desarrollo de la actividad 1
# Preparar dimension y mostrar la distribución por tipo.
# Comprobar formato y unicidad del código.
# Desarrollo de la actividad 1
divipola["codigo_municipio"] = divipola["Código Municipio"].str.strip()

es_cinco_digitos = divipola["codigo_municipio"].str.fullmatch(r"[0-9]{5}").all()
es_unica = divipola["codigo_municipio"].is_unique

print(f"¿Todos los códigos tienen 5 dígitos numéricos?: {es_cinco_digitos}")
print(f"¿La columna de código de municipio es única?: {es_unica}")
print(f"Total de entidades en DIVIPOLA: {len(divipola)}")

print("\nDistribución por Tipo:")
print(divipola["Tipo: Municipio / Isla / Área no municipalizada"].value_counts())

dimension = divipola[[
    "codigo_municipio",
    "Nombre Departamento",
    "Nombre Municipio",
    "Tipo: Municipio / Isla / Área no municipalizada"
]].copy()

display(dimension.head())

¿Todos los códigos tienen 5 dígitos numéricos?: True
¿La columna de código de municipio es única?: True
Total de entidades en DIVIPOLA: 1122

Distribución por Tipo:
Tipo: Municipio / Isla / Área no municipalizada
Municipio                 1103
Área no municipalizada      18
Isla                         1
Name: count, dtype: int64


,codigo_municipio,Nombre Departamento,Nombre Municipio,Tipo: Municipio / Isla / Área no municipalizada
0,05001,ANTIOQUIA,MEDELLÍN,Municipio
1,05002,ANTIOQUIA,ABEJORRAL,Municipio
2,05004,ANTIOQUIA,ABRIAQUÍ,Municipio
3,05021,ANTIOQUIA,ALEJANDRÍA,Municipio
4,05030,ANTIOQUIA,AMAGÁ,Municipio


**Interpretación del resultado:**

La tabla dimension se consolida a partir de DIVIPOLA garantizando la estructura única de las 1.122 entidades territoriales registradas en el catálogo de corte 2024. Se comprobó mediante expresiones regulares que todos los códigos tienen exactamente cinco dígitos (preservando ceros a la izquierda) y son completamente unicos (is_unique = True), clasificándose en Municipios, Áreas no municipalizadas e Islas.

## 2. Códigos EVA sin correspondencia · 15 minutos

Preparar `codigo_municipio` desde `Código Dane municipio` y `anio` desde `Año` de EVA. Conservar originales; los códigos vacíos o mal formados deben quedar identificados para revisión. Completar ceros solo en textos numéricos válidos de menor longitud. Crear combinaciones distintas de código y año y realizar un anti-join frente a dimension. Guardar el resultado en `sin_correspondencia`.

📘 **Apoyo:** `.str.fullmatch`, `.where`, `.str.zfill`, `drop_duplicates` y `merge(how="left", indicator=True, validate="many_to_one")`.

✅ **Comprobación:** El código municipal es la clave de unión. El año se conserva en el diagnóstico, pero no se une contra el corte 2024 de DIVIPOLA. El listado se muestra aunque quede vacío.

In [3]:
# Desarrollo de la actividad 2
# Preparar código y año en eva.
# Obtener sin_correspondencia con código municipal y año.
# Desarrollo de la actividad 2
eva["codigo_municipio_raw"] = eva["Código Dane municipio"].str.strip()
eva["anio"] = pd.to_numeric(eva["Año"], errors="coerce").astype("Int64")

es_numerico = eva["codigo_municipio_raw"].str.fullmatch(r"[0-9]+")
eva["codigo_municipio"] = eva["codigo_municipio_raw"].where(
    ~es_numerico, 
    eva["codigo_municipio_raw"].str.zfill(5)
)

eva_comb = eva[["codigo_municipio", "anio"]].drop_duplicates().copy()

merge_check = eva_comb.merge(
    dimension[["codigo_municipio"]],
    on="codigo_municipio",
    how="left",
    indicator=True,
    validate="many_to_one"
)

sin_correspondencia = merge_check[merge_check["_merge"] == "left_only"][["codigo_municipio", "anio"]].copy()

print(f"Total combinaciones (código, año) en EVA: {len(eva_comb)}")
print(f"Combinaciones sin correspondencia en DIVIPOLA: {len(sin_correspondencia)}")
display(sin_correspondencia.head())

Total combinaciones (código, año) en EVA: 7691
Combinaciones sin correspondencia en DIVIPOLA: 0


,codigo_municipio,anio


**Interpretación del resultado:**

El tratamiento con zfill(5) ajusta los códigos numéricos más cortos agregando los ceros a la izquierda para emparejarlos con la llave DANE de 5 dígitos. La consulta mediante anti-join (left_only) identifica los registros o combinaciones de EVA cuyo código territorial no encuentra equivalencia en el catálogo maestro DIVIPOLA 2024.

## 3. Selección para el indicador · 10 minutos

Convertir `Área cosechada` y `Producción` a `area_ha` y `produccion_t`. Seleccionar primero `Cultivo == "Papa"` y año 2024 en `papa_2024`. De ese conjunto, conservar en `apta` las filas con área positiva y producción no negativa. Contar las filas excluidas del indicador dentro de papa_2024.

📘 **Apoyo:** `pd.to_numeric`, `.eq`, `.gt`, `.ge`, `.loc` y `.copy()`.

✅ **Comprobación:** El total de papa_2024 es igual a las filas aptas más las excluidas del indicador. La ausencia de una medida no se reemplaza por cero.

In [4]:
# Desarrollo de la actividad 3
# Crear papa_2024 y apta.
# Mostrar las exclusiones del indicador y conciliar los conteos.
# Desarrollo de la actividad 3
eva["area_ha"] = pd.to_numeric(eva["Área cosechada"], errors="coerce")
eva["produccion_t"] = pd.to_numeric(eva["Producción"], errors="coerce")

papa_2024 = eva[(eva["Cultivo"] == "Papa") & (eva["anio"] == 2024)].copy()

mask_apta = (papa_2024["area_ha"] > 0) & (papa_2024["produccion_t"] >= 0)
apta = papa_2024.loc[mask_apta].copy()
excluidas = papa_2024.loc[~mask_apta].copy()

total_papa_2024 = len(papa_2024)
total_apta = len(apta)
total_excluidas = len(excluidas)

print(f"Total registros Papa 2024: {total_papa_2024}")
print(f"Registros aptos para indicador: {total_apta}")
print(f"Registros excluidos: {total_excluidas}")

assert total_papa_2024 == (total_apta + total_excluidas), "Error en conciliación de filas."
print("✅ Conciliación de filas correcta.")

Total registros Papa 2024: 831
Registros aptos para indicador: 779
Registros excluidos: 52
✅ Conciliación de filas correcta.


**Interpretación del resultado:**

Se aislaron los reportes de Papa para el año 2024. El filtrado separa estrictamente las filas aptas para el indicador (con área cosechada positiva) de aquellas inconsistentes o ausentes. La suma de las filas aptas y excluidas equivale al 100% de los registros de Papa 2024, asegurando que no se imputó ni sustituyó ningún dato faltante por cero.

## 4. Rendimiento agregado · 15 minutos

Agrupar apta por código municipal, cultivo, estado físico del cultivo y año. Crear `rendimiento` con cantidad de registros, suma de producción y suma de área cosechada. Calcular `rendimiento_t_ha` dividiendo ambas sumas.

📘 **Apoyo:** `groupby(..., as_index=False, dropna=False)`, `agg`, `size` y `sum`.

✅ **Comprobación:** Cada grupo tiene área total positiva. La suma de los conteos por grupo coincide con len(apta). El indicador se calcula como suma de producción / suma de área, conservando cultivo y estado físico en la agrupación.

In [8]:
# Desarrollo de la actividad 4
# Crear rendimiento con los agregados y rendimiento_t_ha.
# Comprobar denominadores y cantidad de registros agrupados.
# Desarrollo de la actividad 4
col_estado = "Estado físico del cultivo" if "Estado físico del cultivo" in apta.columns else "Estado fisico del cultivo"

rendimiento = (
    apta.groupby(["codigo_municipio", "Cultivo", col_estado, "anio"], as_index=False, dropna=False)
    .agg(
        registros=("area_ha", "size"),
        produccion_total_t=("produccion_t", "sum"),
        area_total_ha=("area_ha", "sum")
    )
)

rendimiento["rendimiento_t_ha"] = rendimiento["produccion_total_t"] / rendimiento["area_total_ha"]

print(f"Total registros en apta: {len(apta)}")
print(f"Suma de registros en grupos: {rendimiento['registros'].sum()}")
assert len(apta) == rendimiento["registros"].sum(), "La cantidad de registros agrupados difiere de apta."
assert (rendimiento["area_total_ha"] > 0).all(), "Existen grupos con área total no positiva."

print("✅ Verificación de denominadores y conteos agregados exitosa.")
display(rendimiento.head())
display(rendimiento["rendimiento_t_ha"].describe().to_frame())

Total registros en apta: 779
Suma de registros en grupos: 779
✅ Verificación de denominadores y conteos agregados exitosa.


,codigo_municipio,Cultivo,Estado físico del cultivo,anio,registros,produccion_total_t,area_total_ha,rendimiento_t_ha
0,05001,Papa,En fresco,2024,2,1280.0,80.0,16.000000
1,05002,Papa,En fresco,2024,2,1650.0,165.0,10.000000
2,05004,Papa,En fresco,2024,4,15.0,5.0,3.000000
3,05079,Papa,En fresco,2024,2,268.0,34.0,7.882353
4,05086,Papa,En fresco,2024,3,10550.0,905.0,11.657459


,rendimiento_t_ha
count,292.000000
mean,17.978680
std,6.587110
min,3.000000
25%,14.363880
50%,18.000000
75%,21.724340
max,48.763699


**Interpretación del resultado:**

El rendimiento municipal en toneladas por hectárea (rendimiento_t_ha) se calculó de forma técnicamente correcta mediante la razón de agregados: (Σ producción / Σ área). Esto garantiza que el denominador represente la superficie real consolidada por municipio y evita los sesgos de estimación que genera promediar rendimientos individuales desproporcionados.

## 5. Unión con nombres y clasificación · 10 minutos

Relacionar rendimiento con dimension mediante un left merge por codigo_municipio. Declarar `validate="many_to_one"` e incorporar un indicador de correspondencia. Guardar el resultado en `rendimiento_municipal` y mostrar los códigos sin nombre territorial, si existen.

📘 **Apoyo:** `merge`, `validate`, `indicator` y `len`.

✅ **Comprobación:** La cantidad de filas antes y después de la unión debe ser idéntica. Un fallo de cardinalidad debe investigarse; no se resuelve eliminando arbitrariamente claves de la dimensión.

In [9]:
# Desarrollo de la actividad 5
# Unir rendimiento con dimension.
# Verificar la conservación de filas y presentar la cobertura de la unión.
# Desarrollo de la actividad 5
rendimiento_municipal = rendimiento.merge(
    dimension,
    on="codigo_municipio",
    how="left",
    validate="many_to_one",
    indicator=True
)

print(f"Filas antes del merge (rendimiento): {len(rendimiento)}")
print(f"Filas después del merge (rendimiento_municipal): {len(rendimiento_municipal)}")
assert len(rendimiento) == len(rendimiento_municipal), "El merge alteró el número de filas."

sin_nombre = rendimiento_municipal[rendimiento_municipal["_merge"] == "left_only"]
print(f"Códigos municipales sin correspondencia en DIVIPOLA: {len(sin_nombre)}")
if len(sin_nombre) > 0:
    display(sin_nombre)

display(rendimiento_municipal.head())

Filas antes del merge (rendimiento): 292
Filas después del merge (rendimiento_municipal): 292
Códigos municipales sin correspondencia en DIVIPOLA: 0


,codigo_municipio,Cultivo,Estado físico del cultivo,anio,registros,produccion_total_t,area_total_ha,rendimiento_t_ha,Nombre Departamento,Nombre Municipio,Tipo: Municipio / Isla / Área no municipalizada,_merge
0,05001,Papa,En fresco,2024,2,1280.0,80.0,16.000000,ANTIOQUIA,MEDELLÍN,Municipio,both
1,05002,Papa,En fresco,2024,2,1650.0,165.0,10.000000,ANTIOQUIA,ABEJORRAL,Municipio,both
2,05004,Papa,En fresco,2024,4,15.0,5.0,3.000000,ANTIOQUIA,ABRIAQUÍ,Municipio,both
3,05079,Papa,En fresco,2024,2,268.0,34.0,7.882353,ANTIOQUIA,BARBOSA,Municipio,both
4,05086,Papa,En fresco,2024,3,10550.0,905.0,11.657459,ANTIOQUIA,BELMIRA,Municipio,both


**Interpretación del resultado:**

El left merge enriqueció la tabla de rendimiento con los nombres oficiales de departamento, municipio y tipo de entidad. La regla validate="many_to_one" confirmó que la dimensión es una clave primaria limpia, preservando exactamente el número original de filas de la agregación sin duplicaciones no deseadas.

## 6. Exportación e interpretación · 10 minutos

Exportar rendimiento_municipal a `OUT / "rendimiento_municipal.csv"` y sin_correspondencia a `OUT / "codigos_sin_correspondencia.csv"`, ambos sin índice. Explicar por qué se dividen sumas en lugar de promediar rendimientos individuales y qué limitación introduce comparar EVA con un catálogo territorial de corte 2024.

📘 **Apoyo:** `to_csv` y los resultados de las actividades anteriores.

✅ **Comprobación:** El archivo de códigos conserva sus cabeceras aunque no tenga filas. La explicación distingue una coincidencia de códigos de una equivalencia histórica de límites.

In [16]:
# Desarrollo de la actividad 6
# Exportar las dos tablas.
# Explicar el denominador y la limitación temporal en la celda siguiente.
# Desarrollo de la actividad 6
import os

ruta_rendimiento = os.path.join(OUT, "rendimiento_municipal.csv")
ruta_sin_corresp = os.path.join(OUT, "codigos_sin_correspondencia.csv")

rendimiento_municipal.to_csv(ruta_rendimiento, index=False, encoding="utf-8")
sin_correspondencia.to_csv(ruta_sin_corresp, index=False, encoding="utf-8")

print(f"✅ Exportado exitosamente: {ruta_rendimiento}")
print(f"✅ Exportado exitosamente: {ruta_sin_corresp}")



✅ Exportado exitosamente: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03/rendimiento_municipal.csv
✅ Exportado exitosamente: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03/codigos_sin_correspondencia.csv


**Interpretación del resultado:**
Se dividen sumas en lugar de promediar rendimientos individuales dado que del cálculo del rendimiento (Denominador agregado vs Promedio de rendimientos), se deben dividir las sumas agregadas (Σ producción / Σ área) lo que es matemáticamente correcto porque asigna el peso adecuado a cada predio o reporte según su extensión. Promediar de forma simple los rendimientos pondera por igual una parcela pequeña que un área industrial extensa, introduciendo sesgos estadísticos.
Por otro lado la limitación de comparar fuentes históricas con DIVIPOLA 2024, es que usar un catálogo territorial de corte 2024 para auditar registros históricos de EVA implica que municipios creados, segregados o renombrados en años pasados pueden aparecer sin correspondencia o mal categorizados si los límites o códigos DANE cambiaron a lo largo del tiempo.

## Entrega

El notebook completado se conserva como `soluciones/PD03_integracion_territorial.ipynb`. Las salidas regenerables se guardan en `kit/salidas/soluciones/PD03/`.

✅ Deben quedar visibles el control de la dimensión, el anti-join, el conteo de exclusiones, la conciliación de grupos y la conservación de filas en el merge.

⚠️ Una coincidencia de códigos no prueba estabilidad de límites municipales a través del tiempo. El rendimiento agregado se interpreta para el cultivo y estado físico seleccionados; no debe sumarse con rendimientos de otros cultivos.